# ECE1508 — Unified MoE / AAG Benchmark

**Author: Mohammad Al Dridi**

Runs every model variant through one identical pipeline and produces the
results table and figures for the report.

**Colab Pro setup — do this first:**

1. Runtime → Change runtime type → **L4 GPU** (24 GB). A100 also works and is
   faster, but burns compute units about 3x quicker. T4 works too, just slower.
2. Runtime → **enable background execution**, so the sweep survives closing
   the tab. This is the main reason Pro is worth it here.

Results are written to Google Drive after every variant, so a disconnect never
loses completed work.


## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Clone the project repo. Use a token or SSH if the repo is private.
REPO = "https://github.com/tmrcnl/ECE1508-DeepGenerativeModels.git"
BRANCH = "mohammad/benchmark-and-aag"

import os, shutil
if os.path.exists("ECE1508-DeepGenerativeModels"):
    shutil.rmtree("ECE1508-DeepGenerativeModels")
!git clone --branch {BRANCH} {REPO}
%cd ECE1508-DeepGenerativeModels/mohammad

In [ ]:
!pip install -q -r requirements.txt

## 2. Smoke test

Two variants, 32 training examples, sequence length 64. Numbers are
meaningless — this only proves the pipeline runs before you spend GPU hours.

In [ ]:
!python -m moe_bench.runner --preset smoke --variants smoke --out results_smoke

## 3. The core sweep

Five variants — dense baseline, both of Tamara's MoEs, AAG at 1 and 8 chunks —
all on the same frozen split with the same hyperparameters.

Roughly 20–30 minutes per variant on a T4, so budget ~2 hours. `results.json`
is rewritten after every variant, so a disconnect keeps the completed ones.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Write results to Drive so a runtime disconnect does not lose them.
OUT = "/content/drive/MyDrive/ece1508_results/core"
!mkdir -p "{OUT}"

In [ ]:
!python -m moe_bench.runner     --preset budget     --variants core     --out "{OUT}"     --batch-size 8 --grad-accum 2 --epochs 1

## 4. The AAG scaling curve

chunks = 1, 2, 4, 8, 16 → 16 to 3.4e19 virtual experts, at constant storage
and constant per-token compute. This is the figure that tests the proposal's
central claim.

Add `--gradient-checkpointing` if 16 chunks runs out of memory.

In [ ]:
OUT_SCALING = "/content/drive/MyDrive/ece1508_results/scaling"
!mkdir -p "{OUT_SCALING}"

!python -m moe_bench.runner     --preset budget     --variants scaling     --out "{OUT_SCALING}"     --batch-size 8 --grad-accum 2 --epochs 1

## 5. Tables and figures

Writes `results.md`, `results.tex` (paste straight into the Prism report) and
the PNGs for the slides.

In [ ]:
!python -m moe_bench.report "{OUT}/results.json" --out "{OUT}"

In [ ]:
from IPython.display import Image, display
for name in ["fig_perplexity", "fig_capacity", "fig_routing"]:
    display(Image(f"{OUT}/{name}.png"))

In [ ]:
!python -m moe_bench.report "{OUT_SCALING}/results.json" --out "{OUT_SCALING}"
display(Image(f"{OUT_SCALING}/fig_scaling.png"))

## 6. Evaluating the team's existing checkpoints

The sweep above retrains everything under one budget, which is the controlled
comparison. To also score Tamara's full-Alpaca checkpoints on the same frozen
split, point `load_checkpoint` at her Drive folder.

In [ ]:
import torch
from moe_bench import builders, data, metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = builders.load_tokenizer()
_, eval_ds = data.build_splits(tokenizer, data.PRESETS["budget"])

CHECKPOINT = "/content/drive/MyDrive/gpt2-moe-alpaca/model.safetensors"

model = builders.build("moe-top1")
info = builders.load_checkpoint(model, CHECKPOINT)
print("missing:", len(info["missing"]), " unexpected:", len(info["unexpected"]))

model.to(device)
quality = metrics.evaluate_perplexity(model, eval_ds, device)
print("perplexity:", quality["perplexity"])
print("routing:", metrics.routing_health(quality["_routing_counts"])["entropy_ratio"])